<a href="https://colab.research.google.com/github/gabriel-tfg/TFM_RAG_Pipeline/blob/main/PruebasTFM7_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers

In [2]:
pip install faiss-cpu

1) Query configuration

In [3]:
import requests
import xml.etree.ElementTree as ET
import json
import re
from collections import defaultdict

QUERY_1 = """
"meta-analysis"[Publication Type] AND ("Menopause"[MeSH]) AND
("Quality of Life"[MeSH]
OR "Sleep Wake Disorders"[MeSH]
OR "Stress, Psychological"[MeSH]
OR "Fatigue"[MeSH]
OR "Anxiety"[MeSH]
OR "Cognition Disorders"[MeSH]
OR "Cognitive Dysfunction"[MeSH]
OR "Executive Function"[MeSH]
OR "Memory"[MeSH]
OR cognit*[tiab]
OR fatigue[tiab])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_2 = """
("meta-analysis"[Publication Type] AND "menopause/psychology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_3 = """
("meta-analysis"[Publication Type] AND "menopause/physiology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

ACTIVE_QUERY = QUERY_1   # cambiar a QUERY_2 o QUERY_3 para pruebas
RETMAX = 100

2) Pubmed search helpers

In [4]:
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ELINK_URL   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

def pubmed_search(query, retmax=20):
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json"
    }
    data = requests.get(ESEARCH_URL, params=params).json()
    return data["esearchresult"]["idlist"]

def pubmed_fetch_xml(pmids):
    if not pmids:
        return None
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }
    xml_text = requests.get(EFETCH_URL, params=params).content
    return ET.fromstring(xml_text)

def parse_pubmed_articles(root):
    docs = []
    if root is None:
        return docs

    for i, article in enumerate(root.findall(".//PubmedArticle")):
        pmid_elem = article.find(".//PMID")
        title_elem = article.find(".//ArticleTitle")
        abstract_elems = article.findall(".//Abstract/AbstractText")

        pmid = pmid_elem.text.strip() if pmid_elem is not None and pmid_elem.text else None
        if title_elem is None:
            continue

        title = "".join(title_elem.itertext()).strip()
        abstract = " ".join("".join(a.itertext()).strip() for a in abstract_elems) if abstract_elems else ""

        text = f"{title}\n\n{abstract}".strip()

        docs.append({
            "id": f"meta_{pmid}" if pmid else f"meta_{i}",
            "pmid": pmid,
            "type": "meta-analysis",
            "title": title,
            "text": text
        })

    return docs

# Buscar metaanálisis
meta_pmids = pubmed_search(ACTIVE_QUERY, retmax=RETMAX)
print("Meta-analysis PMIDs:", len(meta_pmids))

meta_root = pubmed_fetch_xml(meta_pmids)
meta_docs = parse_pubmed_articles(meta_root)

print("Meta-analysis docs:", len(meta_docs))
print(meta_docs[0]["title"] if meta_docs else "No docs")

Meta-analysis PMIDs: 32
Meta-analysis docs: 32
Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis.


3) Link PubMed -> PMC

In [5]:
import requests
import time

ID_CONVERTER_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"

def get_pmcid_for_pmid(pmid, tool="rag_pipeline", email="tu_email@ejemplo.com"):
    params = {
        "ids": str(pmid),
        "format": "json",
        "tool": tool,
        "email": email
    }

    try:
        r = requests.get(ID_CONVERTER_URL, params=params, timeout=20)
        r.raise_for_status()

        # Ver qué tipo de contenido devuelve realmente
        content_type = r.headers.get("Content-Type", "")

        if "json" not in content_type.lower():
            print(f"[Aviso] PMID {pmid}: la respuesta no es JSON.")
            print("Content-Type:", content_type)
            print("Respuesta:", r.text[:300])
            return None

        if not r.text.strip():
            print(f"[Aviso] PMID {pmid}: respuesta vacía.")
            return None

        data = r.json()
        records = data.get("records", [])
        if not records:
            print(f"[Aviso] PMID {pmid}: sin records.")
            return None

        pmcid = records[0].get("pmcid")
        return pmcid

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] PMID {pmid}: {e}")
        return None
    except ValueError as e:
        print(f"[Error JSON] PMID {pmid}: {e}")
        print("Respuesta recibida:", r.text[:300])
        return None

meta_to_pmc = {}
for d in meta_docs:
    if d.get("pmid"):
        meta_to_pmc[d["pmid"]] = get_pmcid_for_pmid(d["pmid"])
        time.sleep(0.3)

print(meta_to_pmc)

{'41903824': None, '41531400': 'PMC12835450', '41448220': None, '41401249': None, '41401244': None, '41341454': 'PMC12669000', '41161257': 'PMC12597306', '41129871': None, '41066270': None, '40971657': None, '40742785': None, '40718787': 'PMC12296567', '40575963': None, '40288155': None, '40231842': 'PMC12599504', '40224208': 'PMC11992344', '40105038': None, '39987726': None, '39919442': None, '39856668': 'PMC11762881', '39145901': None, '38977980': 'PMC11229230', '38619017': None, '38563867': None, '38489855': None, '37960761': 'PMC10637479', '37945913': None, '37697662': 'PMC10594314', '37477236': None, '36147572': 'PMC9486389', '34758929': None, '33880736': 'PMC8595144'}


4) PMC XML retrieval + reference extraction

In [6]:
from collections import defaultdict
import requests
import xml.etree.ElementTree as ET
import time
import re

PMC_OAI_URL = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"

def fetch_pmc_xml_oai(pmcid):
    if not pmcid:
        return None

    pmc_num = pmcid.replace("PMC", "").strip()

    params = {
        "verb": "GetRecord",
        "identifier": f"oai:pubmedcentral.nih.gov:{pmc_num}",
        "metadataPrefix": "pmc"
    }

    try:
        r = requests.get(PMC_OAI_URL, params=params, timeout=30)
        r.raise_for_status()

        if not r.text.strip():
            print(f"[Aviso] {pmcid}: respuesta vacía")
            return None

        return r.text

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] {pmcid}: {e}")
        return None

def extract_article_xml_from_oai(oai_xml_text):
    if not oai_xml_text:
        return None

    try:
        root = ET.fromstring(oai_xml_text)
    except Exception as e:
        print(f"[Error XML OAI] {e}")
        return None

    ns = {
        "oai": "http://www.openarchives.org/OAI/2.0/"
    }

    metadata = root.find(".//oai:metadata", ns)
    if metadata is None or len(metadata) == 0:
        print("[Aviso] No se encontró bloque <metadata> en la respuesta OAI")
        return None

    article_elem = list(metadata)[0]

    try:
        return ET.tostring(article_elem, encoding="unicode")
    except Exception as e:
        print(f"[Error serializando article XML] {e}")
        return None

def extract_references_from_pmc_xml(article_xml_text):
    refs = []
    if not article_xml_text:
        return refs

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando article XML] {e}")
        return refs

    for i, ref in enumerate(root.findall(".//{*}ref")):
        ref_text = " ".join(t.strip() for t in ref.itertext() if t and t.strip())

        pmid = None
        doi = None
        year = None
        first_author = None

        ref_id = ref.attrib.get("id", f"ref_{i}")

        for pubid in ref.findall(".//{*}pub-id"):
            id_type = pubid.attrib.get("pub-id-type", "").lower()
            if pubid.text:
                if id_type == "pmid" and pmid is None:
                    pmid = pubid.text.strip()
                elif id_type == "doi" and doi is None:
                    doi = pubid.text.strip()

        year_match = re.search(r"\b(19|20)\d{2}\b", ref_text)
        if year_match:
            year = year_match.group(0)

        surname_elem = ref.find(".//{*}surname")
        if surname_elem is not None and surname_elem.text:
            first_author = surname_elem.text.strip().lower()

        refs.append({
            "ref_id": ref_id,
            "pmid": pmid,
            "doi": doi,
            "year": year,
            "first_author": first_author,
            "ref_text": ref_text
        })

    return refs

4.1) Included study detection (heuristic)

In [7]:
import xml.etree.ElementTree as ET

def extract_candidate_sections(article_xml_text):
    sections = []
    if not article_xml_text:
        return sections

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para secciones] {e}")
        return sections

    for sec in root.findall(".//{*}sec"):
        title_elem = sec.find("./{*}title")
        title = ""
        if title_elem is not None:
            title = " ".join(t.strip() for t in title_elem.itertext() if t and t.strip()).lower()

        sec_text = " ".join(t.strip() for t in sec.itertext() if t and t.strip())
        sections.append({
            "title": title,
            "text": sec_text.lower()
        })

    return sections

def extract_candidate_tables(article_xml_text):
    tables = []
    if not article_xml_text:
        return tables

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para tablas] {e}")
        return tables

    for tbl in root.findall(".//{*}table-wrap"):
        label_elem = tbl.find(".//{*}label")
        caption_elem = tbl.find(".//{*}caption")

        label = " ".join(label_elem.itertext()).strip().lower() if label_elem is not None else ""
        caption = " ".join(caption_elem.itertext()).strip().lower() if caption_elem is not None else ""
        text = " ".join(t.strip() for t in tbl.itertext() if t and t.strip()).lower()

        tables.append({
            "label": label,
            "caption": caption,
            "text": text
        })

    return tables

def score_reference_as_included(ref, sections, tables):
    score = 0
    evidence = []

    first_author = ref.get("first_author")
    year = ref.get("year")

    if ref.get("pmid"):
        score += 2
        evidence.append("has_pmid")

    relevant_titles = [
        "included studies",
        "characteristics of included studies",
        "study characteristics",
        "included study",
        "results",
        "studies included",
        "study selection"
    ]

    for sec in sections:
        sec_title = sec["title"]
        sec_text = sec["text"]

        title_is_relevant = any(key in sec_title for key in relevant_titles)

        if first_author and year and first_author in sec_text and year in sec_text:
            if title_is_relevant:
                score += 5
                evidence.append(f"section:{sec_title}")
            else:
                score += 2
                evidence.append("author_year_in_section")

    table_keywords = [
        "included",
        "characteristics",
        "study",
        "studies"
    ]

    for tbl in tables:
        tbl_meta = f"{tbl['label']} {tbl['caption']}"
        tbl_text = tbl["text"]
        table_is_relevant = any(k in tbl_meta for k in table_keywords)

        if first_author and year and first_author in tbl_text and year in tbl_text:
            if table_is_relevant:
                score += 6
                evidence.append(f"table:{tbl_meta[:80]}")
            else:
                score += 3
                evidence.append("author_year_in_table")

    return score, evidence

def extract_included_study_candidates(article_xml_text, min_score=5):
    refs = extract_references_from_pmc_xml(article_xml_text)
    sections = extract_candidate_sections(article_xml_text)
    tables = extract_candidate_tables(article_xml_text)

    included = []
    all_scored = []

    for ref in refs:
        score, evidence = score_reference_as_included(ref, sections, tables)

        item = {
            **ref,
            "score": score,
            "evidence": evidence
        }
        all_scored.append(item)

        if score >= min_score:
            included.append(item)

    included = sorted(included, key=lambda x: x["score"], reverse=True)
    all_scored = sorted(all_scored, key=lambda x: x["score"], reverse=True)

    return included, all_scored

4.2) Test included-study detection + build mappings

In [8]:
from collections import defaultdict
import time

test_pmcid = "PMC12835450"

oai_xml = fetch_pmc_xml_oai(test_pmcid)
print(oai_xml[:500] if oai_xml else "No OAI XML")

article_xml = extract_article_xml_from_oai(oai_xml)
print(article_xml[:500] if article_xml else "No article XML")

refs = extract_references_from_pmc_xml(article_xml)
print("N refs:", len(refs))
print("First refs:", refs[:3])

included, all_scored = extract_included_study_candidates(article_xml, min_score=5)
print("Included candidates:", len(included))
print("Top included candidates:", included[:5])

meta_included_candidates = defaultdict(list)
meta_all_scored_refs = defaultdict(list)

for d in meta_docs:
    pmid = d.get("pmid")
    pmcid = meta_to_pmc.get(pmid)

    if pmcid:
        oai_xml = fetch_pmc_xml_oai(pmcid)
        article_xml = extract_article_xml_from_oai(oai_xml)

        included, all_scored = extract_included_study_candidates(article_xml, min_score=5)

        meta_included_candidates[pmid] = included
        meta_all_scored_refs[pmid] = all_scored

        print(f"Meta PMID {pmid}: included candidates = {len(included)} / all refs = {len(all_scored)}")
        time.sleep(0.34)

print("Example meta PMID:", next(iter(meta_included_candidates)) if meta_included_candidates else "No meta")




<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
    <responseDate>2026-05-18T11:23:40Z</responseDate>
    <request verb="GetRecord" identifier="oai:pubmedcentral.nih.gov:12835450" metadataPrefix="pmc">https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/</request>
    
    <GetRecord>
        <record>
    <header>
    <identifier
<ns0:article xmlns:ns0="https://jats.nlm.nih.gov/ns/archiving/1.4/" xmlns:ns2="http://www.niso.org/schemas/ali/1.0/" xmlns:ns3="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="https://jats.nlm.nih.gov/ns/archiving/1.4/ https://jats.nlm.nih.gov/archiving/1.4/xsd/JATS-archivearticle1-4.xsd" xml:lang="en" article-type="research-article" dtd-version="1.4"><ns0:processing-meta base-tagset="archiving" mathml-version="3.0" table-model="xhtml" tag

5) Collect cited PMIDs from references

In [9]:
included_pmids = set()

for meta_pmid, refs in meta_included_candidates.items():
    for ref in refs:
        if ref.get("pmid"):
            included_pmids.add(str(ref["pmid"]))

included_pmids = sorted(included_pmids)

print("Included-study candidate PMIDs found:", len(included_pmids))
print(included_pmids[:10])

Included-study candidate PMIDs found: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']


5.2) Save included PMIDs + prepare new PMIDs

In [10]:
import json

# 1) Guardar todos los PMIDs incluidos (heurísticos)
with open("included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(included_pmids, f, ensure_ascii=False, indent=2)

# 2) PMIDs semilla (metaanálisis)
seed_pmids = {
    str(d["pmid"])
    for d in meta_docs
    if d.get("pmid")
}

# 3) Filtrar: quitar los metaanálisis
new_included_pmids = [
    str(pmid) for pmid in included_pmids
    if str(pmid) not in seed_pmids
]

# 4) Ordenar
new_included_pmids = sorted(set(new_included_pmids))

# 5) Guardar
with open("new_included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(new_included_pmids, f, ensure_ascii=False, indent=2)

print("Seed PMIDs:", len(seed_pmids))
print("All included PMIDs:", len(included_pmids))
print("New included PMIDs:", len(new_included_pmids))
print(new_included_pmids[:10])

Seed PMIDs: 32
All included PMIDs: 114
New included PMIDs: 114
['10183310', '11457767', '14412840', '14706761', '1484915', '15511602', '16024761', '16522699', '16568338', '16737531']


6) Fetch cited papers

In [11]:
included_root = pubmed_fetch_xml(new_included_pmids[:200])
included_docs_raw = parse_pubmed_articles(included_root)

included_docs = []
for d in included_docs_raw:
    included_docs.append({
        "id": f"included_{d['pmid']}" if d.get("pmid") else d.get("id"),
        "pmid": d.get("pmid"),
        "type": "included-study-candidate",
        "title": d.get("title"),
        "text": d.get("text")
    })

print("Included-study candidate docs:", len(included_docs))
print(included_docs[0]["title"] if included_docs else "No docs")

Included-study candidate docs: 114
The MOS 36-item short form health survey. A conceptual analysis.


7) Final corpus + repository files

In [12]:
import json

seen = set()
docs = []

for d in meta_docs + included_docs:
    pmid = d.get("pmid")
    if pmid and pmid not in seen:
        docs.append(d)
        seen.add(pmid)

print("TOTAL DOCS IN CORPUS:", len(docs))

# guardar corpus
with open("corpus_meta_plus_included.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

# guardar mapping meta → included candidates
with open("meta_included_map.json", "w", encoding="utf-8") as f:
    json.dump(dict(meta_included_candidates), f, ensure_ascii=False, indent=2)

print("Saved corpus_meta_plus_included.json")
print("Saved meta_included_map.json")

TOTAL DOCS IN CORPUS: 146
Saved corpus_meta_plus_included.json
Saved meta_included_map.json


8) Chunking

In [13]:
import re

def chunk_text_sentences(text, chunk_chars=1200, overlap_chars=200):
    if not text or not text.strip():
        return []

    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    cur = ""

    for s in sents:
        s = s.strip()
        if not s:
            continue

        # si una sola frase ya supera el tamaño, la troceamos de forma defensiva
        if len(s) > chunk_chars:
            if cur:
                chunks.append(cur)
                cur = ""

            start = 0
            while start < len(s):
                end = start + chunk_chars
                piece = s[start:end]
                chunks.append(piece)
                start += max(1, chunk_chars - overlap_chars)
            continue

        if len(cur) + len(s) + 1 <= chunk_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s

    if cur:
        chunks.append(cur)

    out = []
    for i, c in enumerate(chunks):
        if i == 0:
            out.append(c)
        else:
            prev = out[-1]
            overlap = prev[-overlap_chars:] if len(prev) > overlap_chars else prev
            out.append((overlap + " " + c).strip())

    return out


# reconstruir chunks con metadatos útiles para RAG
chunks = []

for d in docs:
    doc_text = d.get("text", "")
    doc_chunks = chunk_text_sentences(doc_text, chunk_chars=1200, overlap_chars=200)

    for i, c in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{d['id']}_chunk_{i}",
            "source_id": d.get("id"),
            "pmid": d.get("pmid"),
            "type": d.get("type"),
            "source_type_priority": 2 if d.get("type") == "meta-analysis" else 1,
            "title": d.get("title"),
            "chunk_index": i,
            "text": c
        })

print("N chunks:", len(chunks))
print("Chunk example:", chunks[0]["text"][:300] if chunks else "No chunks")

N chunks: 295
Chunk example: Can massage ameliorate menopausal and postmenopausal symptoms in women? A systematic review and meta-analysis. This systematic review and meta-analysis study explores the effect of massage on comprehensive menopause symptoms, including vasomotor, physical, psychological, sexual, sleep, anxiety, depr


9) Embeddings + FAISS + retrieve()

In [14]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1) Modelo de embeddings
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2) Textos de los chunks
texts = [c["text"] for c in chunks]

# 3) Crear embeddings
emb = embed_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")

# 4) Crear índice FAISS con similitud coseno vía inner product
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Embeddings shape:", emb.shape)
print("FAISS index size:", index.ntotal)

def keyword_score(query, title, text):
    """
    Re-ranking ligero por palabras clave biomédicas.
    Premia coincidencias específicas entre query expandida y título/texto.
    """
    q = query.lower()
    title_l = (title or "").lower()
    text_l = (text or "").lower()
    full = title_l + " " + text_l

    score = 0.0

    keyword_groups = {
        "cbt_insomnia": [
            "cbt-i", "cbti", "cognitive behavioral therapy", "cognitive behavioural therapy",
            "insomnia", "insomnia severity", "sleep quality", "sleep restriction",
            "sleep hygiene", "isi", "psqi"
        ],
        "hot_flashes": [
            "hot flashes", "hot flushes", "vasomotor", "night sweats",
            "menopausal symptoms", "climacteric symptoms"
        ],
        "mental_health": [
            "depression", "depressive", "anxiety", "stress", "psychological",
            "mood", "irritability", "mental health", "quality of life"
        ],
        "mindfulness": [
            "mindfulness", "mbsr", "mindfulness-based stress reduction",
            "psychoeducation", "mind-body"
        ],
        "cardio": [
            "cardiovascular", "heart disease", "cardiometabolic", "coronary",
            "lipid", "blood pressure", "metabolic"
        ],
        "hormone_therapy": [
            "hormone therapy", "menopausal hormone therapy", "mht", "hrt",
            "estrogen", "oestradiol", "estradiol"
        ]
    }

    # Si la query contiene términos de un grupo, premiar documentos que contengan términos de ese grupo
    for group_terms in keyword_groups.values():
        query_hits = sum(1 for term in group_terms if term in q)
        if query_hits > 0:
            title_hits = sum(1 for term in group_terms if term in title_l)
            text_hits = sum(1 for term in group_terms if term in text_l)

            score += 0.035 * title_hits
            score += 0.012 * text_hits

    # Premiar fuertemente si una palabra muy específica aparece en el título
    specific_terms = [
        "cbt-i", "cbti", "cognitive behavioral therapy", "cognitive behavioural therapy",
        "hot flashes", "hot flushes", "vasomotor",
        "mindfulness", "depression", "anxiety",
        "cardiovascular", "heart disease"
    ]

    for term in specific_terms:
        if term in q and term in title_l:
            score += 0.06

    return score


def noise_penalty(query, title, text):
    """
    Penaliza papers que aparecen mucho pero no son centrales para ciertas preguntas.
    """
    q = query.lower()
    title_l = (title or "").lower()
    text_l = (text or "").lower()
    full = title_l + " " + text_l

    penalty = 0.0

    # El paper de massage estaba contaminando muchas respuestas.
    # Solo permitimos que destaque si la pregunta va de massage.
    if "massage" in full and "massage" not in q and "masaje" not in q:
        penalty += 0.10

    # Penalizar cuestionarios si la pregunta pide tratamiento o mecanismo
    if any(term in full for term in ["questionnaire", "scale", "instrument", "validation"]):
        if any(term in q for term in [
            "treatment", "therapy", "manage", "improve", "mejor", "tratamiento",
            "qué puedo hacer", "sofocos", "insomnio", "mindfulness", "cbt"
        ]):
            penalty += 0.05

    # Penalizar ruido de IA/app si no se pregunta por IA/app
    if any(term in full for term in ["artificial intelligence", "digital health", "app"]):
        if not any(term in q for term in ["app", "ai", "ia", "artificial intelligence", "digital"]):
            penalty += 0.08

    return penalty


def retrieve(query, k=10, initial_k=40, max_per_doc=1, use_type_priority=True, use_keyword_rerank=True):
    """
    Retrieval FAISS + re-ranking biomédico ligero.
    """
    if not query or not query.strip():
        return []

    if len(chunks) == 0:
        return []

    initial_k = min(initial_k, len(chunks))
    k = min(k, len(chunks))

    q = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = index.search(q, initial_k)

    candidates = []

    for score, i in zip(scores[0], idxs[0]):
        ch = chunks[i]

        title = ch.get("title", "")
        text = ch.get("text", "")

        adjusted_score = float(score)

        # Prioridad por tipo de fuente
        if use_type_priority:
            adjusted_score += 0.02 * float(ch.get("source_type_priority", 1))

        # Re-ranking por keywords
        k_score = 0.0
        n_penalty = 0.0

        if use_keyword_rerank:
            k_score = keyword_score(query, title, text)
            n_penalty = noise_penalty(query, title, text)
            adjusted_score += k_score
            adjusted_score -= n_penalty

        candidates.append({
            "score": float(score),
            "keyword_score": float(k_score),
            "noise_penalty": float(n_penalty),
            "adjusted_score": float(adjusted_score),
            "chunk_id": ch.get("chunk_id"),
            "source_id": ch.get("source_id"),
            "pmid": ch.get("pmid"),
            "type": ch.get("type"),
            "title": title,
            "chunk_index": ch.get("chunk_index"),
            "text": text
        })

    candidates = sorted(candidates, key=lambda x: x["adjusted_score"], reverse=True)

    results = []
    seen_docs = {}

    for cand in candidates:
        doc_id = cand["source_id"]
        count = seen_docs.get(doc_id, 0)

        if count >= max_per_doc:
            continue

        results.append(cand)
        seen_docs[doc_id] = count + 1

        if len(results) >= k:
            break

    return results


# 6) Test retrieval rápido
res = retrieve(
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    k=8,
    initial_k=20,
    max_per_doc=1,
    use_type_priority=True
)

for r in res:
    print(
        f"score={r['score']:.3f} | adj={r['adjusted_score']:.3f} | "
        f"type={r['type']} | pmid={r['pmid']} | source={r['source_id']}"
    )
    print("title:", r["title"])
    print(r["text"][:220], "\n---\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeddings shape: (295, 384)
FAISS index size: 295
score=0.435 | adj=0.455 | type=included-study-candidate | pmid=14412840 | source=included_14412840
title: Contemporary therapy of the menopausal syndrome.
Contemporary therapy of the menopausal syndrome. 
---

score=0.390 | adj=0.430 | type=meta-analysis | pmid=37477236 | source=meta_37477236
title: The impact of menopause education on quality of life among menopausal women: a systematic review with meta-analysis.
The impact of menopause education on quality of life among menopausal women: a systematic review with meta-analysis. A systematic review with meta-analysis was conducted to establish the impact of menopause health educat 
---

score=0.375 | adj=0.415 | type=meta-analysis | pmid=39145901 | source=meta_39145901
title: Prevalence of poor sleep quality during menopause: a meta-analysis.
Prevalence of poor sleep quality during menopause: a meta-analysis. Numerous researches have demonstrated that sleep quality deteriorates during 

In [15]:
# Pruebas pre-modelo

queries = [
    "How does CBT-I affect sleep quality in menopausal women?",
    "Does CBT-I improve insomnia severity during menopause?",
    "What evidence exists for CBT-I in postmenopausal women with vasomotor symptoms?"
]

for q in queries:
    print("\nQUERY:", q)
    res = retrieve(q, k=5, initial_k=15, max_per_doc=1, use_type_priority=True)
    for r in res:
        print(f"score={r['score']:.3f} | type={r['type']} | pmid={r['pmid']}")
        print("title:", r["title"])
        print("---")


QUERY: How does CBT-I affect sleep quality in menopausal women?
score=0.711 | type=meta-analysis | pmid=41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
---
score=0.747 | type=included-study-candidate | pmid=30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
---
score=0.804 | type=included-study-candidate | pmid=33818845
title: The effect of group cognitive behavioural therapy for insomnia in postmenopausal women.
---
score=0.758 | type=meta-analysis | pmid=39145901
title: Prevalence of poor sleep quality during menopause: a meta-analysis.
---
score=0.687 | type=meta-analysis | pmid=40575963
title: Sleep quality in perimenopausal and postmenopausal women: which exercise therapy is the most effective? A systemat

10) Generative model: Gemma

In [16]:
from huggingface_hub import login
login()  # Esto abrirá un navegador donde te pedirá tus credenciales de Hugging Face

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "google/gemma-2-2b-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
)

model.to(device)
model.eval()

Device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

11) Full RAG (build_prompt + answer_rag)

In [18]:
import torch, re

def build_context(results, max_chars=3200, max_per_doc=1):
    """
    Construye el contexto para RAG asegurándose de que solo se incluyan fragmentos relevantes.
    """
    context_parts = []
    total_chars = 0
    seen_docs = {}

    for r in results:
        doc_id = r.get("source_id")

        count = seen_docs.get(doc_id, 0)
        if count >= max_per_doc:
            continue

        # Solo agregar textos relevantes que contengan la información más precisa
        piece = (
            f"[TITLE] {r.get('title', 'Unknown title')}\n"
            f"[PMID] {r.get('pmid', 'NA')}\n"
            f"[TYPE] {r.get('type', 'unknown')}\n"
            f"[TEXT] {r.get('text', '')}\n"
        )

        if total_chars + len(piece) > max_chars:
            break

        context_parts.append(piece)
        total_chars += len(piece)
        seen_docs[doc_id] = count + 1

    return "\n\n".join(context_parts)


def expand_query_for_retrieval(question):
    """
    Expande preguntas naturales a una query biomédica más útil para FAISS.
    Mantiene la pregunta original y añade sinónimos técnicos en inglés.
    """
    q = question.lower()

    extra_terms = []

    # Sueño / insomnio / CBT-I
    if any(term in q for term in [
        "cbt", "cbt-i", "cognitive behavioral", "cognitive behavioural",
        "insomnia", "sleep", "sueño", "dormir", "duermo", "insomnio",
        "desvelo", "me despierto", "calidad del sueño",
        "terapia cognitivo-conductual", "terapia cognitivo conductual",
        "tcc", "tcc-i", "conductual", "cognitivo-conductual"
    ]):
        extra_terms.append(
            "CBT-I cognitive behavioral therapy cognitive behavioural therapy "
            "insomnia sleep quality sleep disturbance insomnia severity "
            "menopausal women postmenopausal women perimenopausal women "
            "ISI PSQI sleep hygiene sleep restriction therapy"
            "CBT cognitive behavioral therapy cognitive behavioural therapy CBT-Meno menopause symptoms menopausal symptoms"
        )

    # Sofocos / calor / síntomas vasomotores
    if any(term in q for term in [
        "hot flash", "hot flashes", "hot flush", "hot flushes",
        "vasomotor", "sofoco", "sofocos", "calor", "sudor", "sudores",
        "sudoración", "night sweats", "sudores nocturnos"
    ]):
        extra_terms.append(
            "hot flashes hot flushes vasomotor symptoms night sweats "
            "menopause postmenopause perimenopause menopausal symptoms "
            "treatment intervention hormone therapy nonhormonal therapy "
            "exercise mindfulness acupuncture CBT"
        )

    # Salud mental / ansiedad / depresión / estrés
    if any(term in q for term in [
        "mental health", "depression", "depressive", "anxiety", "stress",
        "psychological", "mood", "irritable", "irritability",
        "salud mental", "depresión", "depresion", "ansiedad", "estrés",
        "estres", "ánimo", "animo", "irritable", "triste", "emocional"
    ]):
        extra_terms.append(
            "menopause mental health depression anxiety stress psychological symptoms "
            "mood irritability depressive symptoms postmenopausal women "
            "perimenopausal women quality of life mindfulness CBT mind-body therapies"
        )

    # Calidad de vida / síntomas generales
    if any(term in q for term in [
        "quality of life", "wellbeing", "well-being", "symptoms",
        "calidad de vida", "bienestar", "síntomas", "sintomas",
        "normal", "etapa", "menopausia", "perimenopausia", "postmenopausia"
    ]):
        extra_terms.append(
            "menopause menopausal symptoms climacteric symptoms quality of life "
            "Menopause Rating Scale MRS MENQOL vasomotor psychological physical sexual symptoms "
            "perimenopausal women postmenopausal women"
        )

    # Cognición / memoria / niebla mental
    if any(term in q for term in [
        "cognition", "cognitive", "memory", "brain fog", "concentration",
        "memoria", "concentración", "concentracion", "niebla mental",
        "despistes", "olvidos", "pensar", "lenta", "cabeza"
    ]):
        extra_terms.append(
            "menopause cognition cognitive function memory brain fog concentration "
            "executive function cognitive complaints perimenopause postmenopause"
        )

    # Cuerpo / cambios físicos / piel / peso
    if any(term in q for term in [
        "skin", "dryness", "weight", "body", "fat", "hair",
        "piel", "sequedad", "peso", "engorda", "barriga",
        "cuerpo", "físico", "fisico", "pelo", "calva"
    ]):
        extra_terms.append(
            "menopause physical symptoms body composition weight gain abdominal fat "
            "skin dryness hair loss postmenopausal women estrogen decline"
        )

    # Terapia hormonal / medicación / suplementos
    if any(term in q for term in [
        "hormone therapy", "hormonal therapy", "mht", "hrt",
        "medication", "supplement", "cancer risk",
        "terapia hormonal", "tratamiento hormonal", "medicación", "medicacion",
        "suplementos", "cáncer", "cancer", "medicamentos"
    ]):
        extra_terms.append(
            "menopause hormone therapy menopausal hormone therapy MHT HRT "
            "nonhormonal treatment medication supplements cancer risk "
            "clinical guidelines postmenopausal women"
        )

    # IA / app / límites tecnológicos
    if any(term in q for term in [
        "app", "ai", "ia", "inteligencia artificial", "technology", "tecnología",
        "tecnologia", "diagnose", "diagnóstico", "diagnostico"
    ]):
        extra_terms.append(
            "digital health artificial intelligence health app menopause symptom tracking "
            "clinical decision support patient education medical advice diagnosis limitations"
        )

    if extra_terms:
        return question + " " + " ".join(extra_terms)

    return question

def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    )

    # Mover inputs al mismo dispositivo que el modelo
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=1
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return answer

def clean_answer(text):
    if not text:
        return "No hay suficiente información relevante en el contexto recuperado."

    # 1) Limpiar tokens especiales y markdown básico
    bad_tokens = [
        "<|assistant|>", "<|user|>", "<|system|>",
        "[/INST]", "[INST]", "<s>", "</s>"
    ]

    for tok in bad_tokens:
        text = text.replace(tok, " ")

    text = text.replace("**", " ")
    text = text.replace("*", " ")
    text = text.replace("[TITLE]", "el estudio")
    text = text.replace("[PMID]", "PMID")
    text = text.replace("[TYPE]", "tipo")
    text = text.replace("[TEXT]", "texto")

    text = text.strip()

    # 2) Si el modelo repite secciones del prompt, quedarnos con la parte útil
    section_markers = [
        "Answer in English",
        "Answer in Spanish",
        "Responde en español",
        "Respuesta:",
        "Answer:",
    ]

    for marker in section_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    # Eliminar etiquetas internas si aparecen al principio
    leading_markers = [
        "Pregunta del usuario:",
        "Pregunta:",
        "User question:",
        "Question:",
        "Contexto científico:",
        "Contexto:",
        "Scientific context:",
        "Context:"
    ]

    for marker in leading_markers:
        text = text.replace(marker, " ")

    # 3) Quitar frases introductorias/meta
    meta_patterns = [
        r"^bas[aá]ndose en el contexto proporcionado,?\s*",
        r"^seg[uú]n el contexto proporcionado,?\s*",
        r"^de acuerdo con el contexto proporcionado,?\s*",
        r"^con base en el contexto proporcionado,?\s*",
        r"^el contexto indica que\s*",
        r"^la respuesta es que\s*",
        r"^la respuesta es\s*",
        r"^en respuesta a la pregunta,?\s*",
        r"^para responder a la pregunta,?\s*",
        r"^based on the provided context,?\s*",
        r"^according to the provided context,?\s*",
    ]

    for pat in meta_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE).strip()

    # 4) Separar líneas y eliminar preguntas repetidas o cabeceras
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    clean_lines = []

    for line in lines:
        low = line.lower().strip()

        # eliminar líneas que son solo símbolos
        if re.fullmatch(r"[\*\-\#\:\s]+", line):
            continue

        # eliminar cabeceras
        if low in ["contexto", "respuesta", "answer", "context", "question", "pregunta"]:
            continue

        # eliminar preguntas repetidas cortas
        if line.endswith("?") and len(line) < 220:
            continue

        clean_lines.append(line)

    text = " ".join(clean_lines).strip()

    # 5) Normalizar espacios y puntuación
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = text.strip(" -:;,.")
    text = text.strip()

    # 6) Cortar frases genéricas incompletas o inútiles
    weak_or_empty = [
        "la evidencia científica disponible sugiere que",
        "la evidencia disponible sugiere que",
        "la evidencia científica sugiere que",
        "la evidencia sugiere que",
        "según el contexto",
        "no hay suficiente información",
        "not enough information",
        "no hay suficiente información.",
        "not enough information.",
    ]

    if text.lower().strip() in weak_or_empty:
        return "No hay suficiente información relevante en el contexto recuperado."

    # Si empieza con una frase incompleta tipo "La evidencia sugiere que."
    text = re.sub(
        r"^(La evidencia científica disponible sugiere que|La evidencia disponible sugiere que|La evidencia científica sugiere que|La evidencia sugiere que)\.?\s*$",
        "No hay suficiente información relevante en el contexto recuperado.",
        text,
        flags=re.IGNORECASE
    )

    # 7) Eliminar duplicados de frases
    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen = set()
    unique = []

    for s in sentences:
        s = s.strip()
        if not s:
            continue

        key = s.lower()
        if key not in seen:
            seen.add(key)
            unique.append(s)

    text = " ".join(unique).strip()

    # 8) Si queda algo demasiado corto o basura, devolver fallback
    if len(text) < 25:
        return "No hay suficiente información relevante en el contexto recuperado."

    if re.fullmatch(r"[\W_]+", text):
        return "No hay suficiente información relevante en el contexto recuperado."

    # 9) Asegurar punto final
    if text and text[-1] not in ".!?":
        text += "."

    # Capitalización inicial
    text = text[0].upper() + text[1:]

    return text


def answer_rag(question, k=4, initial_k=25, max_per_doc=1, max_new_tokens=150):
    retrieval_query = expand_query_for_retrieval(question)

    contexts = retrieve(
        retrieval_query,
        k=k,
        initial_k=initial_k,
        max_per_doc=max_per_doc,
        use_type_priority=True
    )

    context_text = build_context(
        contexts,
        max_chars=3000,
        max_per_doc=max_per_doc
    )

    prompt = f"""Eres un asistente de investigación biomédica especializado en salud hormonal femenina, perimenopausia, menopausia y postmenopausia.

    Debes responder a la pregunta del usuario usando el contexto científico proporcionado.
    El contexto contiene títulos y resúmenes de artículos científicos, normalmente en inglés.
    Puedes sintetizar la evidencia disponible, aunque el contexto esté en inglés.
    No inventes datos que no estén apoyados por el contexto.
    Si el contexto responde solo parcialmente, explica qué está apoyado por la evidencia y qué queda incierto.
    Solo responde "No hay suficiente información" si el contexto recuperado no tiene relación clara con la pregunta.

    Pregunta del usuario:
    {question}

    Contexto científico:
    {context_text}

    Responde en español, en 2-4 frases claras:
    """

    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)

    return {
        "question": question,
        "retrieval_query": retrieval_query,
        "answer": answer,
        "contexts": contexts,
        "prompt": prompt
    }


# Quick domain test

# Lista de preguntas de ejemplo
questions = [
    # Español
    "¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño y la gravedad del insomnio en mujeres menopáusicas?",
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    "¿Cómo afecta la menopausia a la salud mental?",
    "¿La menopausia es igual para todas las mujeres?",
    "¿El estrés puede empeorar los síntomas de la menopausia?",
    "¿Cómo afectan los cambios hormonales a la calidad del sueño en mujeres postmenopáusicas?",
    "¿Cuál es el mejor tratamiento para el insomnio durante la menopausia?",
    "¿La menopausia aumenta el riesgo de enfermedad cardiovascular?",
    "¿Puede el mindfulness ayudar con los síntomas de la menopausia?",
    "¿Es eficaz la terapia cognitivo-conductual en mujeres menopáusicas?",

    # Inglés
    "Does CBT-I improve sleep quality and insomnia severity in menopausal women?",
    "Can stress contribute to menopause symptoms?"
]

# Ejecutamos el modelo para cada pregunta
for q in questions:
    rag_result = answer_rag(
        q,
        k=4,
        initial_k=25,
        max_per_doc=1,
        max_new_tokens=150
    )

    print("\n" + "="*80)
    print("QUESTION:\n", rag_result["question"])

    if "retrieval_query" in rag_result:
        print("\nRETRIEVAL QUERY:\n", rag_result["retrieval_query"])

    print("\nANSWER:\n", rag_result["answer"])

    print("\nTOP CONTEXTS:")
    for c in rag_result["contexts"][:4]:
        print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")


QUESTION:
 ¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño y la gravedad del insomnio en mujeres menopáusicas?

RETRIEVAL QUERY:
 ¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño y la gravedad del insomnio en mujeres menopáusicas? CBT-I cognitive behavioral therapy cognitive behavioural therapy insomnia sleep quality sleep disturbance insomnia severity menopausal women postmenopausal women perimenopausal women ISI PSQI sleep hygiene sleep restriction therapyCBT cognitive behavioral therapy cognitive behavioural therapy CBT-Meno menopause symptoms menopausal symptoms digital health artificial intelligence health app menopause symptom tracking clinical decision support patient education medical advice diagnosis limitations

ANSWER:
 Explicación: El contexto indica que una meta-análisis encontró que la terapia cognitivo-conductual para el insomnio (CBT-I) puede mejorar la calidad del sueño y la gravedad del insomnio en mujeres co

Test retrieval 2

In [19]:
test_queries = [
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    "¿Puede el mindfulness ayudar con los síntomas de la menopausia?",
    "¿La menopausia aumenta el riesgo de enfermedad cardiovascular?",
    "¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño?",
    "Can stress contribute to menopause symptoms?"
]

for q in test_queries:
    expanded = expand_query_for_retrieval(q)

    res = retrieve(
        expanded,
        k=5,
        initial_k=40,
        max_per_doc=1,
        use_type_priority=True,
        use_keyword_rerank=True
    )

    print("\n" + "="*80)
    print("QUESTION:", q)
    print("EXPANDED:", expanded[:300], "...")

    for r in res:
        print(
            f"adj={r['adjusted_score']:.3f} | "
            f"base={r['score']:.3f} | "
            f"kw={r['keyword_score']:.3f} | "
            f"pen={r['noise_penalty']:.3f} | "
            f"type={r['type']} | PMID={r['pmid']}"
        )
        print("title:", r["title"])
        print("---")


QUESTION: ¿Qué puedo hacer con los sofocos durante la menopausia?
EXPANDED: ¿Qué puedo hacer con los sofocos durante la menopausia? hot flashes hot flushes vasomotor symptoms night sweats menopause postmenopause perimenopause menopausal symptoms treatment intervention hormone therapy nonhormonal therapy exercise mindfulness acupuncture CBT menopause menopausal symptoms clim ...
adj=0.915 | base=0.597 | kw=0.298 | pen=0.000 | type=included-study-candidate | PMID=21372745
title: Mindfulness training for coping with hot flashes: results of a randomized trial.
---
adj=0.787 | base=0.578 | kw=0.189 | pen=0.000 | type=included-study-candidate | PMID=39992004
title: Mindfulness for Menopausal Women: Enhancing Quality of Life and Psychological Well-Being Through a Randomized Controlled Intervention.
---
adj=0.757 | base=0.562 | kw=0.155 | pen=0.000 | type=meta-analysis | PMID=39987726
title: Efficacy and safety of fezolinetant and elinzanetant for vasomotor symptoms in postmenopausal women: A

12) Baseline vs RAG comparison

In [20]:
# =========================
# 12) Baseline vs RAG comparison
# =========================

def answer_base(question, max_new_tokens=220):
    prompt = f"""You are a helpful biomedical assistant specializing in female hormonal health,
    menstrual cycle, fertility, perimenopause, menopause, postmenopause, thyroid health, and metabolic changes.
    You are also an expert in symptoms such as sleep disturbances, stress, and other hormonal changes throughout a woman's life.

    Answer the question clearly in 2-4 sentences.
    If you are not sure, say so.

    Question:
    {question}

    Answer:
    """
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)
    return answer

queries = {
    "Incertidumbre diagnóstica": [
        "¿Cómo sé si lo que me está pasando es menopausia o no?",
        "¿Esto que me pasa es por la edad o por otra cosa?",
        "¿Todo lo que me pasa ahora tiene que ver con la menopausia?",
        "¿Cómo diferencio si un síntoma es hormonal o no?",
        "¿Puede ser solo estrés y no menopausia?",
        "¿Estoy en menopausia aunque no tenga todos los síntomas?",
        "¿Esto es normal en esta etapa o debería preocuparme?",
        "¿Hay alguna forma de saber con certeza qué cosas son de la menopausia y cuáles no?",
        "¿Todo lo que me pasa ahora lo estoy relacionando mal con la menopausia?",
        "¿Estoy exagerando o realmente me está pasando algo?"
    ],

    "Necesidad de orientación personalizada": [
        "¿Qué pruebas médicas debería hacerme ahora mismo?",
        "¿Qué tipo de especialista debería consultar?",
        "¿Qué seguimiento debería llevar en esta etapa?",
        "¿Qué analíticas son importantes en esta etapa de la menopausia?",
        "¿Cómo sé si me falta alguna vitamina?",
        "¿Qué debería revisar en mis análisis de sangre?",
        "¿Qué controles debería hacerme de forma preventiva?",
        "¿Cuándo debería hacerme una densitometría?",
        "¿Qué revisiones son realmente necesarias y cuáles no?",
        "¿Cómo puedo saber si estoy bien controlada médicamente?"
    ],

    "Demanda de acompañamiento": [
        "¿Por qué nadie me explica claramente qué me está pasando?",
        "¿Existe alguna forma de tener seguimiento diario de mis síntomas?",
        "¿Cómo sería tener un acompañamiento diario en esta etapa?",
        "¿Qué información necesito para tomar decisiones sobre mi salud?",
        "¿Cómo puedo tener una guía clara de qué hacer en cada momento?",
        "¿Por qué siento que voy a ciegas en este proceso?",
        "¿Cómo puedo organizar toda la información que recibo?",
        "¿Cómo puedo saber qué es importante y qué no?",
        "¿Cómo puedo tener una visión global de mi estado?"
    ],

    "Prevención": [
        "¿Qué debería haber hecho antes para prevenir esto?",
        "¿A qué edad debería haber empezado a cuidarme?",
        "¿Qué puedo hacer ahora para evitar empeorar?",
        "¿Es tarde para empezar a prevenir?",
        "¿Qué hábitos son clave en esta etapa?",
        "¿Cómo puedo prevenir problemas futuros?",
        "¿Qué debería cambiar en mi estilo de vida?",
        "¿Qué cosas empeoran los síntomas sin darme cuenta?",
        "¿Cómo influye la alimentación en esta etapa?",
        "¿Qué debería haber sabido antes?"
    ],

    "Cuerpo y cambios físicos": [
        "¿Por qué me están pasando cosas en el cuerpo que nunca me habían pasado?",
        "¿Es normal que cambie la piel de repente?",
        "¿Por qué tengo picores o cambios en la piel?",
        "¿Es normal tener más sequedad corporal?",
        "¿Por qué mi cuerpo reacciona diferente ahora?",
        "¿Esto que me pasa en el cuerpo es temporal?",
        "¿Mi cuerpo va a volver a ser como antes?",
        "¿Estos cambios son reversibles?",
        "¿Cuánto tiempo duran estos cambios físicos?",
        "¿Qué cambios son normales y cuáles no?",
        "¿De verdad estos cambios físicos dependen tanto de la genética?",
        "¿Me voy a parecer más a mi madre y a mi abuela materna que a la familia de mi padre en cómo llevo la menopausia?",
        "¿Por qué me engorda la barriga más que todo lo demás?",
        "¿Me puedo quedar medio calva o sin pelo por la menopausia?",
        "¿Por qué parece que las famosas no tienen menopausia o se la han “curado”?"
    ],

    "Síntomas específicos": [
        "¿Qué puedo hacer con los sofocos?",
        "¿Hay algo que funcione de verdad para el calor?",
        "¿Por qué tengo tanto calor de repente?",
        "¿Los sofocos se van o se quedan para siempre?",
        "¿Por qué algunas de mis amigas casi no tienen sofocos y yo sí?",
        "¿Qué puedo hacer con el insomnio?",
        "¿Por qué estoy más cansada que antes?",
        "¿Por qué me enfermo más que antes?",
        "¿Es normal tener más infecciones ahora?",
        "¿Por qué tengo dolores nuevos que antes no tenía?",
        "¿Por qué siento que mi sistema inmunológico está peor?",
        "¿Estos síntomas son para siempre o en algún momento aflojan?"
    ],

    "Medicación y sistema sanitario": [
        "¿Por qué me recetan tantas cosas sin explicarme bien para qué son?",
        "¿Estoy tomando demasiados medicamentos?",
        "¿La menopausia se está medicalizando demasiado?",
        "¿Qué alternativas tengo a tomar medicación para todo?",
        "¿Cuándo es realmente necesaria la terapia hormonal?",
        "¿Por qué a unas mujeres se les receta terapia hormonal y a otras no?",
        "¿La medicación para la menopausia puede aumentar el riesgo de cáncer?",
        "Si como bien, ¿de verdad tengo que tomar suplementos?",
        "¿Todas las mujeres deberían tomar suplementos o tratamiento hormonal, salvo contraindicaciones?",
        "¿Tiene límite de tiempo el tratamiento hormonal sustitutivo?",
        "¿Es mejor tratar síntomas sueltos o ir a la causa?",
        "¿Cómo puedo evitar tomar medicamentos innecesarios?",
        "¿Qué decisiones médicas dependen realmente de mí y no solo del médico?"
    ],

    "Límites de la IA": [
        "¿Puede una app decirme qué tomar?",
        "¿Hasta dónde puede ayudar una IA en mi salud?",
        "¿Cómo uso una app sin sustituir al médico de toda la vida?",
        "¿Qué tipo de recomendaciones son seguras en una app?",
        "¿Cómo combinar la información de una app con la atención médica?",
        "¿Qué puede hacer una IA y qué no?",
        "¿Cómo saber cuándo tengo que ir a un profesional de la salud?",
        "¿Puede una app orientarme sin diagnosticarme?",
        "¿Cómo evitar engancharme demasiado a una app o a la IA?",
        "¿Cómo usar la tecnología para cuidarme mejor sin hacer locuras?"
    ],

    "Emoción y estado de ánimo": [
        "¿Cómo afecta la menopausia a mi estado emocional?",
        "¿Por qué estoy más irritable?",
        "¿Por qué me afectan más las cosas que antes?",
        "¿Esto emocional es hormonal o psicológico?",
        "¿Cómo influye el estrés en la menopausia?",
        "¿Por qué siento que no soy la misma?",
        "¿Cómo puedo gestionar mejor mis emociones?",
        "¿Esto es pasajero o se queda?",
        "¿Cómo recuperar algo de estabilidad emocional?",
        "¿Qué relación hay entre menopausia y ansiedad?",
        "¿Por qué estoy tan triste si, en teoría, estaba deseando no volver a tener la regla?",
        "¿Los síntomas de la menopausia se parecen a los de mi madre o mi abuela? Me da miedo acabar igual que ellas."
    ],

    "Sexo, deseo y relaciones": [
        "¿Por qué no tengo ganas de sexo si mi pareja me sigue gustando?",
        "¿A todas las mujeres se les baja la libido en esta etapa?",
        "¿Hasta qué punto la bajada de deseo tiene que ver con el miedo al deterioro físico o a dejar de ser atractiva?",
        "¿La sequedad vaginal es para siempre o se puede mejorar?",
        "¿Hay cosas que ya no se pueden hacer en la cama en esta etapa?",
        "Si cambian las hormonas y sube la testosterona, ¿eso puede cambiar lo que me atrae o de quién me enamoro?"
    ],

    "Niebla mental, memoria y cansancio cognitivo": [
        "¿Es normal que se me olviden palabras que tengo 'en la punta de la lengua' todo el rato?",
        "¿Por qué tengo la sensación de que estoy más lenta para pensar que antes?",
        "¿Esto de la 'niebla mental' es cosa de la menopausia o es que me estoy volviendo torpe?",
        "¿Cómo sé si estos despistes son normales o si deberían preocuparme?",
        "¿Por qué me cuesta más concentrarme en cosas que antes hacía sin pensar?",
        "¿Es normal acabar el día con la cabeza agotada aunque no haya hecho 'tanto'?",
        "¿Voy a recuperar mi agilidad mental o esto se queda así?",
        "¿La menopausia puede afectar a mi memoria a largo plazo o solo a estos despistes del día a día?",
        "¿Qué puedo hacer para mejorar la concentración y no sentir que tengo la cabeza 'nublada'?",
        "¿Cómo explico a los demás que no es que sea despistada, sino que estoy notando cambios mentales reales?"
    ]
}


for topic, topic_queries in queries.items():
    print(f"\nTopic: {topic}\n")
    for q in topic_queries:
        base = answer_base(q, max_new_tokens=220)
        rag_result = answer_rag(
            q,
            k=5,
            initial_k=15,
            max_per_doc=1,
            max_new_tokens=220
        )

        print("\n" + "=" * 80)
        print("QUESTION:", q)

        print("\nBASELINE ANSWER:")
        print(base)

        print("\nRAG ANSWER:")
        print(rag_result["answer"])

        print("\nTOP RAG SOURCES:")
        for c in rag_result["contexts"][:3]:
            print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")


Topic: Incertidumbre diagnóstica


QUESTION: ¿Cómo sé si lo que me está pasando es menopausia o no?

BASELINE ANSWER:
It's great that you're paying attention to your body! Menopause is a natural transition for women, but it can be confusing. Here are some common signs of menopause: irregular periods, hot flashes, night sweats, vaginal dryness, mood swings, and difficulty sleeping. If you experience several of these symptoms consistently over time, it might be worth talking to your doctor about whether or not you're going through menopause.

RAG ANSWER:
La mejor manera de saber si lo que te está pasando es menopausia o no es. Explicación: El artículo sobre la calidad de vida en la menopausia describe un cuestionario para evaluar la calidad de vida en mujeres en la menopausia. Este cuestionario se basa en la experiencia de las mujeres y se ha evaluado su validez y fiabilidad. No hay suficiente información para determinar si lo que te está pasando es menopausia o no.

TOP RAG SOURCES:
- 

KeyboardInterrupt: 